# 02 — Feature Engineering
### E-Commerce Customer Analytics Dashboard

Builds derived columns on top of the cleaned dataset that power the SQL
analysis, Python analysis, and Power BI dashboard:

- `Month`, `Quarter`, `Year` (and a `Year_Month` sort key)
- `Profit_Margin` (per order line)
- Customer-level aggregates: `Customer_Lifetime_Value`, `Average_Order_Value`,
  `Revenue_per_Customer`, `Total_Orders`, `First_Purchase_Date`,
  `Last_Purchase_Date`, `Customer_Tenure_Days`

Output: `ecommerce_features.csv` (order-level, enriched) and
`customer_features.csv` (one row per customer — used heavily in segmentation
and RFM analysis).


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/ecommerce_clean.csv', parse_dates=['Order_Date'])
print(df.shape)
df.head()


(12118, 17)


,Order_ID,Order_Date,Customer_ID,Customer_Name,Gender,Age,City,State,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Sales,Profit,Payment_Method
0,ORD010477,2024-09-08,CUST01027,Aarav Bava,Female,34,Nagpur,Maharashtra,West,Beauty & Personal Care,Perfume,1,1366.44,0.05,1298.12,614.90,Wallet
1,ORD008646,2023-10-05,CUST00581,Bhavya Borah,Male,37,Ranchi,Jharkhand,East,Sports & Fitness,Cricket Bat,2,5190.40,0.00,10380.80,4152.32,Debit Card
2,ORD011576,2023-08-08,CUST00653,Sanya Wable,Female,31,Bhubaneswar,Odisha,East,Books,Self-Help Book,1,722.57,0.00,722.57,289.03,Wallet
3,ORD003500,2022-11-07,CUST00342,Teerth Sawhney,Female,36,Jaipur,Rajasthan,North,Groceries,Dry Fruits Combo,1,345.45,0.00,345.45,96.73,Debit Card
4,ORD008417,2023-10-03,CUST00588,Gauri Som,Female,25,Chennai,Tamil Nadu,South,Groceries,Green Tea Pack,1,346.70,0.00,346.70,79.74,Cash On Delivery


## 1. Date Features

In [2]:
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.month_name()
df['Quarter'] = df['Order_Date'].dt.quarter
df['Year_Quarter'] = df['Year'].astype(str) + '-Q' + df['Quarter'].astype(str)
df['Year_Month'] = df['Order_Date'].dt.to_period('M').astype(str)
df['Day_of_Week'] = df['Order_Date'].dt.day_name()

df[['Order_Date', 'Year', 'Month', 'Month_Name', 'Quarter', 'Year_Quarter', 'Year_Month']].head()


,Order_Date,Year,Month,Month_Name,Quarter,Year_Quarter,Year_Month
0,2024-09-08,2024,9,September,3,2024-Q3,2024-09
1,2023-10-05,2023,10,October,4,2023-Q4,2023-10
2,2023-08-08,2023,8,August,3,2023-Q3,2023-08
3,2022-11-07,2022,11,November,4,2022-Q4,2022-11
4,2023-10-03,2023,10,October,4,2023-Q4,2023-10


## 2. Order-Level Profitability Features

`Profit_Margin` is calculated per order line as Profit / Sales (%).


In [3]:
df['Profit_Margin'] = np.where(df['Sales'] > 0, (df['Profit'] / df['Sales']) * 100, 0).round(2)
df['Gross_Sales'] = (df['Unit_Price'] * df['Quantity']).round(2)
df['Discount_Amount'] = (df['Gross_Sales'] - df['Sales']).round(2)

df[['Order_ID', 'Sales', 'Profit', 'Profit_Margin', 'Gross_Sales', 'Discount_Amount']].head()


,Order_ID,Sales,Profit,Profit_Margin,Gross_Sales,Discount_Amount
0,ORD010477,1298.12,614.90,47.37,1366.44,68.32
1,ORD008646,10380.80,4152.32,40.00,10380.80,0.00
2,ORD011576,722.57,289.03,40.00,722.57,0.00
3,ORD003500,345.45,96.73,28.00,345.45,0.00
4,ORD008417,346.70,79.74,23.00,346.70,0.00


## 3. Customer-Level Aggregates

These metrics are computed **per customer** and then joined back where useful.
- **Customer Lifetime Value (CLV)** — total historical revenue from the customer (simple/historical CLV, standard for a portfolio-level analysis without predictive modeling)
- **Average Order Value (AOV)** — CLV / number of orders, computed both globally and per customer
- **Revenue per Customer** — same as CLV at customer grain; also computed as an overall KPI (Total Revenue / Total Customers)
- **Customer Tenure** — days between first and most recent purchase


In [4]:
customer_agg = df.groupby('Customer_ID').agg(
    Customer_Name=('Customer_Name', 'first'),
    Gender=('Gender', 'first'),
    Age=('Age', 'first'),
    City=('City', 'first'),
    State=('State', 'first'),
    Region=('Region', 'first'),
    Total_Orders=('Order_ID', 'nunique'),
    Total_Quantity=('Quantity', 'sum'),
    Customer_Lifetime_Value=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    First_Purchase_Date=('Order_Date', 'min'),
    Last_Purchase_Date=('Order_Date', 'max'),
).reset_index()

customer_agg['Average_Order_Value'] = (
    customer_agg['Customer_Lifetime_Value'] / customer_agg['Total_Orders']
).round(2)

customer_agg['Customer_Tenure_Days'] = (
    customer_agg['Last_Purchase_Date'] - customer_agg['First_Purchase_Date']
).dt.days

customer_agg['Customer_Lifetime_Value'] = customer_agg['Customer_Lifetime_Value'].round(2)
customer_agg['Total_Profit'] = customer_agg['Total_Profit'].round(2)

customer_agg['Profit_Margin_Pct'] = np.where(
    customer_agg['Customer_Lifetime_Value'] > 0,
    (customer_agg['Total_Profit'] / customer_agg['Customer_Lifetime_Value']) * 100,
    0
).round(2)

# Repeat customer flag
customer_agg['Is_Repeat_Customer'] = customer_agg['Total_Orders'] > 1

print(customer_agg.shape)
customer_agg.sort_values('Customer_Lifetime_Value', ascending=False).head(10)


(1137, 17)


,Customer_ID,Customer_Name,Gender,Age,City,State,Region,Total_Orders,Total_Quantity,Customer_Lifetime_Value,Total_Profit,First_Purchase_Date,Last_Purchase_Date,Average_Order_Value,Customer_Tenure_Days,Profit_Margin_Pct,Is_Repeat_Customer
387,CUST00407,Lajita Walia,Female,23,Delhi,Delhi,North,221,413,2546587.60,565923.13,2022-01-02,2024-12-28,11523.02,1091,22.22,True
929,CUST00982,Ayushman Raja,Male,38,Ahmedabad,Gujarat,West,215,408,2143319.96,457577.67,2022-01-01,2024-12-28,9968.93,1092,21.35,True
348,CUST00365,Ethan Buch,Female,40,Patna,Bihar,East,171,342,1977667.17,451714.96,2022-01-02,2024-12-26,11565.31,1089,22.84,True
661,CUST00696,Krish Rama,Male,30,Bengaluru,Karnataka,South,131,238,1878742.73,377244.51,2022-01-03,2024-12-26,14341.55,1088,20.08,True
726,CUST00767,Girik Nair,Male,33,Chandigarh,Punjab,North,109,229,1719768.05,336194.58,2022-01-11,2024-12-13,15777.69,1067,19.55,True
112,CUST00117,Tristan Iyengar,Male,33,Hyderabad,Telangana,South,97,195,1718395.97,335478.51,2022-01-01,2024-12-21,17715.42,1085,19.52,True
172,CUST00180,Upma Bahl,Female,61,Guwahati,Assam,East,158,284,1496578.34,332684.40,2022-01-12,2024-12-28,9472.01,1081,22.23,True
64,CUST00069,Arya Shere,Male,37,Kochi,Kerala,South,131,239,1429460.19,293497.65,2022-03-05,2024-12-25,10911.91,1026,20.53,True
41,CUST00044,Isaac Patil,Female,30,Dehradun,Uttarakhand,North,85,178,1307365.25,253555.51,2022-01-09,2024-12-24,15380.77,1080,19.39,True
160,CUST00168,Oni Yohannan,Male,52,Mumbai,Maharashtra,West,117,217,1278999.91,263991.11,2022-01-28,2024-12-15,10931.62,1052,20.64,True


## 4. Overall Revenue-per-Customer KPI

In [5]:
total_revenue = df['Sales'].sum()
total_customers = df['Customer_ID'].nunique()
revenue_per_customer_overall = total_revenue / total_customers

print(f"Total Revenue: ₹{total_revenue:,.2f}")
print(f"Total Customers: {total_customers:,}")
print(f"Revenue per Customer (overall): ₹{revenue_per_customer_overall:,.2f}")
print(f"Repeat customers: {customer_agg['Is_Repeat_Customer'].sum():,} "
      f"({customer_agg['Is_Repeat_Customer'].mean()*100:.1f}%)")


Total Revenue: ₹146,834,107.68
Total Customers: 1,137
Revenue per Customer (overall): ₹129,141.70
Repeat customers: 980 (86.2%)


## 5. Join Customer-Level Features Back to Order-Level Table

In [6]:
join_cols = ['Customer_ID', 'Total_Orders', 'Customer_Lifetime_Value',
             'Average_Order_Value', 'Is_Repeat_Customer']

df_features = df.merge(customer_agg[join_cols], on='Customer_ID', how='left')

print(df_features.shape)
df_features.head(3)


(12118, 31)


,Order_ID,Order_Date,Customer_ID,Customer_Name,Gender,Age,City,State,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Sales,Profit,Payment_Method,Year,Month,Month_Name,Quarter,Year_Quarter,Year_Month,Day_of_Week,Profit_Margin,Gross_Sales,Discount_Amount,Total_Orders,Customer_Lifetime_Value,Average_Order_Value,Is_Repeat_Customer
0,ORD010477,2024-09-08,CUST01027,Aarav Bava,Female,34,Nagpur,Maharashtra,West,Beauty & Personal Care,Perfume,1,1366.44,0.05,1298.12,614.90,Wallet,2024,9,September,3,2024-Q3,2024-09,Sunday,47.37,1366.44,68.32,4,11014.37,2753.59,True
1,ORD008646,2023-10-05,CUST00581,Bhavya Borah,Male,37,Ranchi,Jharkhand,East,Sports & Fitness,Cricket Bat,2,5190.40,0.00,10380.80,4152.32,Debit Card,2023,10,October,4,2023-Q4,2023-10,Thursday,40.00,10380.80,0.00,20,159881.56,7994.08,True
2,ORD011576,2023-08-08,CUST00653,Sanya Wable,Female,31,Bhubaneswar,Odisha,East,Books,Self-Help Book,1,722.57,0.00,722.57,289.03,Wallet,2023,8,August,3,2023-Q3,2023-08,Tuesday,40.00,722.57,0.00,48,358731.11,7473.56,True


## 6. Save Engineered Datasets

In [7]:
df_features.to_csv('../data/ecommerce_features.csv', index=False)
customer_agg.to_csv('../data/customer_features.csv', index=False)

print("Saved:")
print(" - ../data/ecommerce_features.csv  (order-level, enriched) ->", df_features.shape)
print(" - ../data/customer_features.csv   (customer-level)        ->", customer_agg.shape)


Saved:
 - ../data/ecommerce_features.csv  (order-level, enriched) -> (12118, 31)
 - ../data/customer_features.csv   (customer-level)        -> (1137, 17)


---
**Next notebook:** `03_exploratory_analysis.ipynb` — revenue trends, regional
performance, product analysis, profitability, and core visualizations.
